####🔹 Where clause "in" in PySpark
.filter(col("ENTITY_ID").isin([row["ENTITY_ID"] for row in rows]))

<b>Similar to </b>
<br>SELECT *
<br>FROM EMP e
<br>where e.dept in (10,20,50)

In [0]:
entity_data = [
    {"ENTITY_ID": 101, "CURRENCY_CODE_BASE_LEGAL_ENT": "USD", "ENTITY FAMILY NAME": "Alpha Fund"},
    {"ENTITY_ID": 102, "CURRENCY_CODE_BASE_LEGAL_ENT": "EUR", "ENTITY FAMILY NAME": "Beta Fund"},
    {"ENTITY_ID": 103, "CURRENCY_CODE_BASE_LEGAL_ENT": "USD", "ENTITY FAMILY NAME": "Gamma Fund"},
    {"ENTITY_ID": 101, "CURRENCY_CODE_BASE_LEGAL_ENT": "USD", "ENTITY FAMILY NAME": "Alpha Fund"}  # duplicate
]


from pyspark.sql import SparkSession
from pyspark.sql.functions import col, lit

# Initialize Spark
spark = SparkSession.builder.appName("EntityMasterExample").getOrCreate()

# Convert list of dicts into DataFrame
entity_master_df = spark.createDataFrame(entity_data)

# Suppose we have a list of rows (like from another query or lookup)
rows = [{"ENTITY_ID": 101}, {"ENTITY_ID": 102}]  # mimic external input

# Use isin with list comprehension
df_entity_master = (
    entity_master_df
    .filter(col("ENTITY_ID").isin([row["ENTITY_ID"] for row in rows]))
    .select("ENTITY_ID", "CURRENCY_CODE_BASE_LEGAL_ENT", "ENTITY FAMILY NAME")
    .withColumn("PARENT_ID", lit(None))
    .withColumn("ENTITY_TYPE", lit("ENTITY"))
    .dropDuplicates(["ENTITY_ID", "CURRENCY_CODE_BASE_LEGAL_ENT"])
)

df_entity_master.show()

####🔹 Inline View in PySpark

<b>✅ 1. Create Sample Data

In [0]:
from pyspark.sql import SparkSession

data = [
    (1, "HR", 4000),
    (2, "HR", 6000),
    (3, "IT", 7000),
    (4, "IT", 3000),
    (5, "Finance", 8000)
]

columns = ["emp_id", "department", "salary"]

df = spark.createDataFrame(data, columns)
df.show()

<b>🔹 Same Example Using SQL (True Inline View)

In [0]:
# Create "employees" table for SQL Query
df.createOrReplaceTempView("employees")

result = spark.sql("""
SELECT *
FROM (
    SELECT department, AVG(salary) AS avg_salary
    FROM employees
    GROUP BY department
) t
WHERE avg_salary > 5000
""")

result.show()

<b>✅ 2. PySpark Code for create Inline View (Subquery Equivalent)

In [0]:
inline_view_df = df.groupBy("department") \
    .avg("salary") \
    .withColumnRenamed("avg(salary)", "avg_salary")

# Filter departments where avg salary > 5000
result_df = inline_view_df.filter("avg_salary > 5000")
result_df.show()


####🔹 Inlive view using two tables

<b>✅ 1. Create Sample Data

In [0]:
# Employee Table
emp_data = [
    (1, "John", 10, 4000),
    (2, "Jane", 10, 6000),
    (3, "Mike", 20, 7000),
    (4, "Sam", 20, 3000),
    (5, "Ravi", 30, 8000)
]

emp_cols = ["emp_id", "emp_name", "dept_id", "salary"]

emp_df = spark.createDataFrame(emp_data, emp_cols)


# Department Table
dept_data = [
    (10, "HR", "India"),
    (20, "IT", "India"),
    (30, "Finance", "India")
]

dept_cols = ["dept_id", "dept_name", "loc"]

dept_df = spark.createDataFrame(dept_data, dept_cols)

<b>🔹 Same Logic Using SQL (True Inline View)

In [0]:
emp_df.createOrReplaceTempView("employee")
dept_df.createOrReplaceTempView("department")

result = spark.sql("""
SELECT d.dept_name, t.avg_salary
FROM (
    SELECT dept_id, AVG(salary) AS avg_salary
    FROM employee
    GROUP BY dept_id
) t
JOIN department d
ON t.dept_id = d.dept_id
WHERE t.avg_salary > 5000
""")

result.show()

<b>🔹 PySpark code for Same Logic.

In [0]:
inline_view_df = emp_df.groupBy("dept_id") \
    .avg("salary") \
    .withColumnRenamed("avg(salary)", "avg_salary")

result_df = inline_view_df \
    .join(dept_df, "dept_id") \
    .filter("avg_salary > 5000")

result_df.show()

In [0]:
%sql
SELECT *
FROM employee e
WHERE e.dept_id IN (
    SELECT d.dept_id
    FROM department d
    WHERE d.loc LIKE 'Ind%'
)

In [0]:
from pyspark.sql.functions import col

result_df = emp_df.alias("e") \
    .join(
        dept_df.filter(col("loc").like("Ind%")).alias("d"), # location India
        col("e.dept_id") == col("d.dept_id"),
        "inner"
    ) \
    .select("e.*")

result_df.show()

#####🔹 Alternative Using .isin() (Closer to SQL IN)

In [0]:
dept_list = [row["dept_id"] for row in dept_df
             .filter(col("loc").like("Ind%"))
             .select("dept_id")
             .collect()]
print(f"dept list: {dept_list}")
result_df = emp_df.filter(col("dept_id").isin(dept_list))

result_df.show()

###🔹 Exists in where Clause Implement
Databricks, <b>EXISTS = left_semi join</b>

In [0]:
%sql
SELECT *
FROM employee e
WHERE EXISTS (
    SELECT 1
    FROM department d
    WHERE e.dept_id = d.dept_id
    AND d.loc LIKE 'Ind%'
)

####👉 EXISTS equivalent to LEFT_SEMI JOIN)

In [0]:
from pyspark.sql.functions import col

result_df = emp_df.join(
    dept_df.filter(col("loc").like("Ind%")),
    emp_df.dept_id == dept_df.dept_id,
    "left_semi"
)

result_df.show()

####👉 NOT EXISTS equivalent to LEFT_ANTI JOIN

In [0]:
result_df = emp_df.join(
    dept_df.filter(col("loc").like("ma%")),
    emp_df.dept_id == dept_df.dept_id,
    "left_anti"
)

result_df.show()

###🔹 Decode Function - similar in Pyspark

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark = SparkSession.builder.getOrCreate()

# Sample data
data = [
    ("2025-03-31", 1000),
    ("2025-06-30", 2000),
    ("2025-09-30", 1500),
    ("2025-12-31", 2500),
]
columns = ["QTR_END_DATE", "QTR_DISTRIBUTION"]

df = spark.createDataFrame(data, columns)

df1 = df.withColumn("DISTRIBUTION_YR", F.quarter("QTR_END_DATE"))
display(df1)

# Define window spec (partition by year, order by quarter)
fund_window_spec = Window.partitionBy(F.year("QTR_END_DATE")).orderBy("QTR_END_DATE")

# Apply the DISTRIBUTION_YR logic
df = df.withColumn(
    "DISTRIBUTION_YR",
    F.when(
        F.quarter("QTR_END_DATE").isin([1, 2, 3]),
        F.col("QTR_DISTRIBUTION")
    ).otherwise(
        F.sum("QTR_DISTRIBUTION").over(fund_window_spec)
    )
)

df.show()



